# 01 - Data ingestion and cleaning
**Goal:** read the source CSV, inspect its quality, and save a reproducible cleaned CSV.
Run all cells in order using **Python (Drug_Assist)**. No API key or database is needed.

**Input:** `data/raw/drug.csv`  
**Output:** `data/processed/drug_clean.csv`  
**Next:** `02_eda.ipynb` or `03_retrieval.ipynb` (each loads this output itself).

## Dataset

- Name: Drug Performance Evaluation
- Source: https://www.kaggle.com/datasets/thedevastator/drug-performance-evaluation
- Listed license: CC0 1.0 Universal
- License URL: https://creativecommons.org/publicdomain/zero/1.0/
- Original data: data/raw/drug.csv
- Cleaned data: data/processed/drug_clean.csv
- Original records: 2,219

## 1. Setup and source data

In [10]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

# Works from the project root or its notebooks folder.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").exists() and (path / "data/raw/drug.csv").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the Drug_Assist project folder.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

RAW_PATH = ROOT / "data/raw/drug.csv"
CLEAN_PATH = ROOT / "data/processed/drug_clean.csv"
print("Python:", sys.executable)
print("Project:", ROOT)

Python: c:\Users\suvra_nw8ieuf\AppData\Local\uv-envs\Drug_Assist\Scripts\python.exe
Project: c:\Users\suvra_nw8ieuf\OneDrive\Desktop\Drug_Assist


In [11]:
df = pd.read_csv(RAW_PATH)
print(f"Raw data: {len(df):,} rows, {len(df.columns)} columns")
display(df.head())
df.info()

Raw data: 2,219 rows, 9 columns


,Condition,Drug,Indication,Type,Reviews,Effective,EaseOfUse,Satisfaction,Information
0,Acute Bacterial Sinusitis,Levofloxacin,On Label,RX,994 Reviews,2.52,3.01,1.84,\r\n\t\t\t\t\tLevofloxacin is used to treat a ...
1,Acute Bacterial Sinusitis,Levofloxacin,On Label,RX,994 Reviews,2.52,3.01,1.84,\r\n\t\t\t\t\tLevofloxacin is used to treat a ...
2,Acute Bacterial Sinusitis,Moxifloxacin,On Label,RX,755 Reviews,2.78,3.00,2.08,\r\n\t\t\t\t\t This is a generic drug. The ave...
3,Acute Bacterial Sinusitis,Azithromycin,On Label,RX,584 Reviews,3.21,4.01,2.57,\r\n\t\t\t\t\tAzithromycin is an antibiotic (m...
4,Acute Bacterial Sinusitis,Azithromycin,On Label,RX,584 Reviews,3.21,4.01,2.57,\r\n\t\t\t\t\tAzithromycin is an antibiotic (m...


<class 'pandas.DataFrame'>
RangeIndex: 2219 entries, 0 to 2218
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Condition     2219 non-null   str    
 1   Drug          2219 non-null   str    
 2   Indication    2219 non-null   str    
 3   Type          2219 non-null   str    
 4   Reviews       2219 non-null   str    
 5   Effective     2219 non-null   float64
 6   EaseOfUse     2219 non-null   float64
 7   Satisfaction  2219 non-null   float64
 8   Information   2219 non-null   str    
dtypes: float64(3), str(6)
memory usage: 156.2 KB


## 2. Missing values and inconsistent labels
These checks inspect the raw data without changing it.

In [12]:
##Check for missing values
# Detect empty strings, spaces, and line breaks without changing df

blank_counts = df.apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and value.strip() == ""
    ).sum()
)

missing_summary = pd.DataFrame({
    "Missing (NaN)": df.isna().sum(),
    "Blank text": blank_counts
})

missing_summary["Total missing or blank"] = missing_summary.sum(axis=1)

display(missing_summary)

,Missing (NaN),Blank text,Total missing or blank
Condition,0,0,0
Drug,0,0,0
Indication,0,31,31
Type,0,7,7
Reviews,0,0,0
Effective,0,0,0
EaseOfUse,0,0,0
Satisfaction,0,0,0
Information,0,0,0


In [13]:
#Check for inconsistencies in the data

for column in ["Condition", "Drug", "Indication", "Type"]:
    labels = pd.DataFrame({
        "original": df[column].dropna().unique()
    })

    labels["normalized"] = (
        labels["original"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .str.casefold()
    )

    variants = labels[
        labels.duplicated("normalized", keep=False)
    ].sort_values("normalized")

    print(f"\n{column}: label variations")
    display(variants)


Condition: label variations


,original,normalized



Drug: label variations


,original,normalized



Indication: label variations


,original,normalized



Type: label variations


,original,normalized


## 3. Duplicates and numeric checks
Only exact copies are removed. Different records for the same drug remain separate.

In [14]:
#Check for duplicates in the data

duplicate_count = df.duplicated().sum()

print(f"Extra duplicate rows: {duplicate_count:,}")
print(f"Rows remaining if removed: {len(df) - duplicate_count:,}")

# Show matching rows together, including the first occurrence
display(
    df[df.duplicated(keep=False)]
    .sort_values(["Condition", "Drug"])
    .head(20)
)

Extra duplicate rows: 466
Rows remaining if removed: 1,753


,Condition,Drug,Indication,Type,Reviews,Effective,EaseOfUse,Satisfaction,Information
13,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,353 Reviews,3.04,3.37,2.34,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
14,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,353 Reviews,3.04,3.37,2.34,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
15,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,353 Reviews,3.04,3.37,2.34,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
16,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,353 Reviews,3.04,3.37,2.34,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
36,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,11 Reviews,2.33,3.67,3.00,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
37,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,11 Reviews,2.33,3.67,3.00,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
38,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,11 Reviews,2.33,3.67,3.00,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
41,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,5 Reviews,3.00,5.00,3.00,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
42,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,5 Reviews,3.00,5.00,3.00,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...
43,Acute Bacterial Sinusitis,Amoxicillin,On Label,RX,5 Reviews,3.00,5.00,3.00,\r\n\t\t\t\t\tAmoxicillin is used to treat a w...


In [15]:
#Are numeric values plausible?
ratings = ["Effective", "EaseOfUse", "Satisfaction"]

display(df[ratings].describe())

print("Examples of review-count labels:")
display(df["Reviews"].drop_duplicates().head(20))

,Effective,EaseOfUse,Satisfaction
count,2219.000000,2219.000000,2219.000000
mean,3.557972,3.958824,3.218774
std,1.113128,1.037877,1.230933
min,1.000000,1.000000,1.000000
25%,3.000000,3.540000,2.400000
50%,3.680000,4.100000,3.130000
75%,4.330000,5.000000,4.000000
max,5.000000,5.000000,5.000000


Examples of review-count labels:


0     994 Reviews
2     755 Reviews
3     584 Reviews
8     437 Reviews
11    361 Reviews
13    353 Reviews
17    222 Reviews
20    140 Reviews
24     72 Reviews
26     43 Reviews
27     40 Reviews
28     23 Reviews
29     20 Reviews
30     19 Reviews
31     15 Reviews
32     13 Reviews
33     12 Reviews
34     11 Reviews
39     10 Reviews
40      5 Reviews
Name: Reviews, dtype: str

In [16]:
review_text = df["Reviews"].astype("string").str.strip()
valid_format = review_text.str.fullmatch(r"\d+\s+Reviews?", na=False)
print("Unexpected review formats:", int((~valid_format).sum()))
display(df.loc[~valid_format, ["Drug", "Reviews"]])

Unexpected review formats: 0


,Drug,Reviews


## 4. Clean and validate
The shared `clean_data` function normalizes whitespace, replaces blank text with missing
values, converts review counts to integers, and removes exact duplicates.
Missing categories remain missing; statistical outliers are not automatically deleted.
These structural checks do not verify whether a drug description is factually correct.

In [17]:
from drug_assist.data import clean_data
df_clean = clean_data(df)
print(f"Rows removed: {len(df) - len(df_clean):,}")
print(f"Cleaned rows: {len(df_clean):,}")
display(df_clean.isna().sum().to_frame("Missing values"))
display(df_clean.head())

Rows removed: 466
Cleaned rows: 1,753


,Missing values
Condition,0
Drug,0
Indication,30
Type,5
Reviews,0
Effective,0
EaseOfUse,0
Satisfaction,0
Information,0


,Condition,Drug,Indication,Type,Reviews,Effective,EaseOfUse,Satisfaction,Information
0,Acute Bacterial Sinusitis,Levofloxacin,On Label,RX,994,2.52,3.01,1.84,Levofloxacin is used to treat a variety of bac...
1,Acute Bacterial Sinusitis,Moxifloxacin,On Label,RX,755,2.78,3.00,2.08,This is a generic drug. The average cash price...
2,Acute Bacterial Sinusitis,Azithromycin,On Label,RX,584,3.21,4.01,2.57,Azithromycin is an antibiotic (macrolide-type)...
3,Acute Bacterial Sinusitis,Amoxicillin-Pot Clavulanate,On Label,RX,437,3.26,3.23,2.42,Amoxicillin/clavulanic acid is a combination p...
4,Acute Bacterial Sinusitis,Levofloxacin,On Label,RX,361,2.44,2.96,1.68,Levofloxacin is used to treat a variety of bac...


## 5. Save the cleaned dataset
Rerunning this cell replaces the derived CSV; the raw CSV is retained.

In [18]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(CLEAN_PATH, index=False)
reloaded = pd.read_csv(CLEAN_PATH, dtype={"Reviews": "Int64"})
pd.testing.assert_frame_equal(df_clean, reloaded, check_dtype=False)
print("Saved and verified:", CLEAN_PATH)

Saved and verified: c:\Users\suvra_nw8ieuf\OneDrive\Desktop\Drug_Assist\data\processed\drug_clean.csv
